In [1]:
import os
import json
import time
import random
from dataclasses import dataclass
from typing import Dict, Tuple, Any, List
import numpy as np
import pandas as pd

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, ConcatDataset
from torchvision import transforms
from torchvision.datasets import ImageFolder

from sklearn.metrics import accuracy_score, classification_report

import optuna


In [3]:

# =========================
# 0) Config
# =========================
@dataclass
class CFG:
    # ---- paths ----
    BASE_DIR: str = "/kaggle/input/datasets/carson650/wildfire/wildfire"  # <-- change if needed
    TRAIN_SPLIT: str = "train/images"
    VAL_SPLIT: str = "valid/images"
    TEST_SPLIT: str = "test"
    OUT_DIR: str = "./runs_flamevision_resnet50"

    # ---- image/model ----
    IMG_SIZE: int = 224
    NUM_CLASSES: int = 2

    # ---- training ----
    SEED: int = 42
    NUM_WORKERS: int = 4
    USE_AMP: bool = True  # mixed precision if CUDA available

    # ---- optuna ----
    N_TRIALS: int = 10
    MAX_EPOCHS_TUNE: int = 10
    EARLY_STOP_PATIENCE: int = 3  # on val macro-F1
    OPTUNA_TIMEOUT_SEC: int = 0   # 0 = no timeout
    STUDY_NAME: str = "flamevision_resnet50_ce"

    # ---- misc ----
    PRINT_EVERY: int = 1

In [4]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Reproducibility (note: may slow training)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def ensure_dir(p: str):
    os.makedirs(p, exist_ok=True)


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
def build_transforms(img_size: int):
    # Train-time augmentation (baseline strong & simple)
    train_tfms = transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406),
                             std=(0.229, 0.224, 0.225)),
    ])

    # Val/Test deterministic transforms
    eval_tfms = transforms.Compose([
        transforms.Resize(int(img_size * 256 / 224)),  # typical: 256 when img_size=224
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406),
                             std=(0.229, 0.224, 0.225)),
    ])
    return train_tfms, eval_tfms


def resolve_split_paths(cfg: CFG) -> Tuple[str, str, str]:
    train_dir = os.path.join(cfg.BASE_DIR, cfg.TRAIN_SPLIT)
    val_dir   = os.path.join(cfg.BASE_DIR, cfg.VAL_SPLIT)
    test_dir  = os.path.join(cfg.BASE_DIR, cfg.TEST_SPLIT)

    for p in [train_dir, val_dir, test_dir]:
        if not os.path.isdir(p):
            raise FileNotFoundError(f"Split dir not found: {p}")

    return train_dir, val_dir, test_dir


def sanity_check_imagefolder(ds: ImageFolder, split_name: str):
    # Ensures ImageFolder sees class subfolders
    if len(ds.classes) < 2:
        raise RuntimeError(
            f"[{split_name}] ImageFolder found {len(ds.classes)} classes. "
            f"Check your folder structure. It must be like split/class_name/*.png"
        )
    if len(ds) == 0:
        raise RuntimeError(f"[{split_name}] Dataset is empty. Check paths and files.")

In [6]:
def compute_class_weights_from_targets(targets: List[int], num_classes: int) -> torch.Tensor:
    """
    Basic inverse-frequency weights:
        w_c = N / (K * count_c)
    """
    counts = np.bincount(np.array(targets), minlength=num_classes).astype(np.float64)
    if (counts == 0).any():
        raise RuntimeError(f"Some class has zero samples: counts={counts.tolist()}")

    N = counts.sum()
    weights = N / (num_classes * counts)
    return torch.tensor(weights, dtype=torch.float32)


def format_metrics(report: Dict[str, Any], class_names: List[str]) -> Dict[str, Any]:
    """
    Returns exactly what you asked:
      - overall: accuracy, recall, f1
      - per-class: precision, recall, f1 for each class
    Here:
      - overall recall = macro avg recall
      - overall f1     = macro avg f1
    """
    overall = {
        "accuracy": float(report["accuracy"]),
        "recall": float(report["macro avg"]["recall"]),  # overall recall (macro)
        "f1": float(report["macro avg"]["f1-score"]),    # overall f1 (macro)
    }

    per_class = {}
    for cname in class_names:
        per_class[cname] = {
            "precision": float(report[cname]["precision"]),
            "recall": float(report[cname]["recall"]),
            "f1": float(report[cname]["f1-score"]),
            "support": int(report[cname]["support"]),
        }

    return {"overall": overall, "per_class": per_class}

In [7]:
@torch.no_grad()
def predict(model: nn.Module, loader: DataLoader, device: torch.device) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        logits = model(xb)
        pred = torch.argmax(logits, dim=1).cpu().numpy()
        ys.append(yb.numpy())
        ps.append(pred)
    return np.concatenate(ys), np.concatenate(ps)


def evaluate(model: nn.Module, loader: DataLoader, device: torch.device, class_names: List[str]) -> Dict[str, Any]:
    y_true, y_pred = predict(model, loader, device)

    rep = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )
    # sklearn report includes accuracy separately but also has "accuracy" key
    # ensure it's consistent:
    rep["accuracy"] = float(accuracy_score(y_true, y_pred))

    return {
        "report_dict": rep,
        "summary": format_metrics(rep, class_names),
        "y_true": y_true,
        "y_pred": y_pred,
    }


def build_resnet50(num_classes: int, device: torch.device) -> nn.Module:
    # torchvision compatibility across versions
    try:
        from torchvision.models import resnet50, ResNet50_Weights
        model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
    except Exception:
        # older torchvision
        from torchvision.models import resnet50
        model = resnet50(pretrained=True)

    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)


def set_backbone_trainable(model: nn.Module, trainable: bool):
    # trainable=False => freeze everything except final fc
    for p in model.parameters():
        p.requires_grad = trainable
    # always keep classifier trainable
    for p in model.fc.parameters():
        p.requires_grad = True


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    use_amp: bool = True,
) -> float:
    model.train()
    scaler = torch.amp.GradScaler("cuda")

    running = 0.0
    n = 0

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda"):
            logits = model(xb)
            loss = criterion(logits, yb)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = xb.size(0)
        running += float(loss.item()) * bs
        n += bs

    return running / max(1, n)


def make_loader(ds, batch_size: int, shuffle: bool, num_workers: int, device: torch.device) -> DataLoader:
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=(device.type == "cuda"),
        drop_last=False,
    )

In [8]:
def run_optuna_tuning(cfg: CFG) -> Dict[str, Any]:
    ensure_dir(cfg.OUT_DIR)
    device = get_device()

    set_seed(cfg.SEED)

    train_tfms, eval_tfms = build_transforms(cfg.IMG_SIZE)
    train_dir, val_dir, test_dir = resolve_split_paths(cfg)

    # Build datasets once per study (transforms fixed)
    ds_train = ImageFolder(train_dir, transform=train_tfms)
    ds_val   = ImageFolder(val_dir, transform=eval_tfms)

    sanity_check_imagefolder(ds_train, "train")
    sanity_check_imagefolder(ds_val, "valid")

    # Ensure consistent class mapping across splits
    if ds_train.class_to_idx != ds_val.class_to_idx:
        raise RuntimeError(
            f"class_to_idx mismatch!\ntrain: {ds_train.class_to_idx}\nvalid: {ds_val.class_to_idx}\n"
            f"Fix folder names so they match exactly."
        )

    class_names = ds_train.classes  # order is consistent with class_to_idx
    class_weights = compute_class_weights_from_targets(ds_train.targets, num_classes=len(class_names)).to(device)

    print("==== Dataset ====")
    print("BASE_DIR:", cfg.BASE_DIR)
    print("train samples:", len(ds_train), "| valid samples:", len(ds_val))
    print("classes:", class_names)
    print("class_to_idx:", ds_train.class_to_idx)
    print("class_weights (for CE):", class_weights.detach().cpu().numpy().round(4).tolist())
    print("device:", device)

    def objective(trial: optuna.Trial) -> float:
        # ---- hyperparams to tune ----
        batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
        optimizer_name = trial.suggest_categorical("optimizer", ["AdamW", "SGD"])
        lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
        weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
        freeze_epochs = trial.suggest_int("freeze_epochs", 0, 3)

        scheduler_name = trial.suggest_categorical("scheduler", ["cosine", "plateau"])
        # Plateau scheduler params
        plateau_factor = trial.suggest_float("plateau_factor", 0.2, 0.8) if scheduler_name == "plateau" else None
        plateau_patience = trial.suggest_int("plateau_patience", 1, 3) if scheduler_name == "plateau" else None

        # SGD extra
        momentum = trial.suggest_float("momentum", 0.8, 0.95) if optimizer_name == "SGD" else None

        max_epochs = cfg.MAX_EPOCHS_TUNE
        patience = cfg.EARLY_STOP_PATIENCE

        # ---- loaders ----
        dl_train = make_loader(ds_train, batch_size=batch_size, shuffle=True,
                               num_workers=cfg.NUM_WORKERS, device=device)
        dl_val   = make_loader(ds_val, batch_size=batch_size, shuffle=False,
                               num_workers=cfg.NUM_WORKERS, device=device)

        # ---- model ----
        model = build_resnet50(num_classes=len(class_names), device=device)

        # Freeze backbone at start if needed
        if freeze_epochs > 0:
            set_backbone_trainable(model, trainable=False)

        # ---- loss ----
        criterion = nn.CrossEntropyLoss(weight=class_weights)

        # ---- optimizer ----
        # Important: pass ALL params, because we will unfreeze later
        if optimizer_name == "AdamW":
            optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        else:
            optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)

        # ---- scheduler ----
        if scheduler_name == "cosine":
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)
        else:
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode="max", factor=plateau_factor, patience=plateau_patience
            )

        # ---- training loop with early-stopping on val macro-F1 ----
        best_val_f1 = -1.0
        best_epoch = 0
        bad = 0

        for epoch in range(1, max_epochs + 1):
            # unfreeze right after freeze_epochs
            if freeze_epochs > 0 and epoch == freeze_epochs + 1:
                set_backbone_trainable(model, trainable=True)

            t0 = time.time()
            train_loss = train_one_epoch(
                model, dl_train, criterion, optimizer, device, use_amp=cfg.USE_AMP
            )

            eval_out = evaluate(model, dl_val, device, class_names)
            val_summary = eval_out["summary"]
            val_f1 = val_summary["overall"]["f1"]  # macro-F1

            # Scheduler step
            if scheduler_name == "plateau":
                scheduler.step(val_f1)
            else:
                scheduler.step()

            # Optuna report + pruning
            trial.report(val_f1, step=epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

            # Early stopping bookkeeping
            improved = val_f1 > best_val_f1 + 1e-6
            if improved:
                best_val_f1 = val_f1
                best_epoch = epoch
                bad = 0
            else:
                bad += 1

            if epoch % cfg.PRINT_EVERY == 0:
                lr_now = optimizer.param_groups[0]["lr"]
                print(
                    f"[trial {trial.number:03d}] epoch {epoch:02d}/{max_epochs} "
                    f"loss={train_loss:.4f} val_acc={val_summary['overall']['accuracy']:.4f} "
                    f"val_recall(macro)={val_summary['overall']['recall']:.4f} val_f1(macro)={val_f1:.4f} "
                    f"lr={lr_now:.2e} time={time.time()-t0:.1f}s"
                )

            if bad >= patience:
                break

        # store best_epoch so we can use it for final train+valid run
        trial.set_user_attr("best_epoch", int(best_epoch))
        return float(best_val_f1)

    # Study setup
    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)
    sampler = optuna.samplers.TPESampler(seed=cfg.SEED)

    study = optuna.create_study(
        direction="maximize",
        sampler=sampler,
        pruner=pruner,
        study_name=cfg.STUDY_NAME
    )

    print("\n==== Optuna tuning start ====")
    study.optimize(
        objective,
        n_trials=cfg.N_TRIALS,
        timeout=(cfg.OPTUNA_TIMEOUT_SEC if cfg.OPTUNA_TIMEOUT_SEC > 0 else None),
        show_progress_bar=True
    )

    best_trial = study.best_trial
    best_params = dict(best_trial.params)
    best_epoch = int(best_trial.user_attrs.get("best_epoch", cfg.MAX_EPOCHS_TUNE))

    print("\n==== Optuna best ====")
    print("best_value (val macro-F1):", best_trial.value)
    print("best_epoch (from tuning):", best_epoch)
    print("best_params:", json.dumps(best_params, indent=2))

    # Save study results
    df_trials = study.trials_dataframe()
    df_trials.to_csv(os.path.join(cfg.OUT_DIR, "optuna_trials.csv"), index=False)

    with open(os.path.join(cfg.OUT_DIR, "optuna_best.json"), "w", encoding="utf-8") as f:
        json.dump(
            {
                "best_value_val_macro_f1": float(best_trial.value),
                "best_epoch": best_epoch,
                "best_params": best_params,
                "class_names": class_names,
                "class_to_idx": ds_train.class_to_idx,
            },
            f,
            indent=2
        )

    return {
        "best_params": best_params,
        "best_epoch": best_epoch,
        "class_names": class_names,
        "class_to_idx": ds_train.class_to_idx
    }

In [9]:
def final_train_and_test(cfg: CFG, best: Dict[str, Any]):
    ensure_dir(cfg.OUT_DIR)
    device = get_device()
    set_seed(cfg.SEED)

    train_tfms, eval_tfms = build_transforms(cfg.IMG_SIZE)
    train_dir, val_dir, test_dir = resolve_split_paths(cfg)

    class_names: List[str] = best["class_names"]
    best_params: Dict[str, Any] = best["best_params"]
    best_epoch: int = int(best["best_epoch"])

    # For final training, we want BOTH train+valid with TRAIN transforms
    ds_train = ImageFolder(train_dir, transform=train_tfms)
    ds_val_as_train = ImageFolder(val_dir, transform=train_tfms)

    # For final testing, we want deterministic eval transforms
    ds_test = ImageFolder(test_dir, transform=eval_tfms)

    # Sanity + class mapping check
    if ds_train.class_to_idx != ds_val_as_train.class_to_idx or ds_train.class_to_idx != ds_test.class_to_idx:
        raise RuntimeError(
            "class_to_idx mismatch across splits. Ensure class folder names match exactly.\n"
            f"train: {ds_train.class_to_idx}\n"
            f"valid: {ds_val_as_train.class_to_idx}\n"
            f"test : {ds_test.class_to_idx}\n"
        )

    # Merge train+valid
    ds_final = ConcatDataset([ds_train, ds_val_as_train])

    # Compute class weights using merged set (optional but consistent)
    # Note: ConcatDataset doesn't have .targets, so we read from components
    merged_targets = list(ds_train.targets) + list(ds_val_as_train.targets)
    class_weights = compute_class_weights_from_targets(merged_targets, num_classes=len(class_names)).to(device)

    # Build loaders
    batch_size = int(best_params["batch_size"])
    dl_final = make_loader(ds_final, batch_size=batch_size, shuffle=True,
                           num_workers=cfg.NUM_WORKERS, device=device)
    dl_test = make_loader(ds_test, batch_size=batch_size, shuffle=False,
                          num_workers=cfg.NUM_WORKERS, device=device)

    # Build model fresh (ImageNet pretrained again)
    model = build_resnet50(num_classes=len(class_names), device=device)

    # Apply freezing strategy from best_params
    freeze_epochs = int(best_params["freeze_epochs"])
    if freeze_epochs > 0:
        set_backbone_trainable(model, trainable=False)

    # Loss
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    # Optimizer
    lr = float(best_params["lr"])
    weight_decay = float(best_params["weight_decay"])
    optimizer_name = best_params["optimizer"]

    if optimizer_name == "AdamW":
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        momentum = float(best_params["momentum"])
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)

    # Scheduler
    scheduler_name = best_params["scheduler"]
    if scheduler_name == "cosine":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, best_epoch))
        plateau = None
    else:
        plateau = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=float(best_params["plateau_factor"]),
            patience=int(best_params["plateau_patience"])
        )
        scheduler = None

    # Final train for exactly best_epoch (as you requested)
    print("\n==== Final training on (train + valid) ====")
    print("final_epochs:", best_epoch, "| freeze_epochs:", freeze_epochs, "| batch_size:", batch_size)
    print("best_params:", json.dumps(best_params, indent=2))

    for epoch in range(1, best_epoch + 1):
        if freeze_epochs > 0 and epoch == freeze_epochs + 1:
            set_backbone_trainable(model, trainable=True)

        train_loss = train_one_epoch(
            model, dl_final, criterion, optimizer, device, use_amp=cfg.USE_AMP
        )

        # No valid set now (by your design). We just step scheduler.
        if scheduler is not None:
            scheduler.step()

        lr_now = optimizer.param_groups[0]["lr"]
        if epoch % cfg.PRINT_EVERY == 0:
            print(f"[final] epoch {epoch:02d}/{best_epoch} loss={train_loss:.4f} lr={lr_now:.2e}")

    # Save final model
    model_path = os.path.join(cfg.OUT_DIR, "final_model_resnet50.pth")
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "class_names": class_names,
            "class_to_idx": ds_train.class_to_idx,
            "best_params": best_params,
            "final_epochs": best_epoch,
            "seed": cfg.SEED,
        },
        model_path
    )
    print("Saved final model:", model_path)

    # Final test evaluation (only once)
    print("\n==== Test evaluation ====")
    test_out = evaluate(model, dl_test, device, class_names)
    summary = test_out["summary"]
    report_dict = test_out["report_dict"]

    # Print exactly what you asked
    print("\n[Overall]")
    print("accuracy:", round(summary["overall"]["accuracy"], 6))
    print("recall  :", round(summary["overall"]["recall"], 6), " (macro)")
    print("f1      :", round(summary["overall"]["f1"], 6), " (macro)")

    print("\n[Per-class]")
    for cname in class_names:
        m = summary["per_class"][cname]
        print(
            f"{cname:>12} | "
            f"P={m['precision']:.6f} R={m['recall']:.6f} F1={m['f1']:.6f} "
            f"(support={m['support']})"
        )

    # Save reports
    with open(os.path.join(cfg.OUT_DIR, "test_report.json"), "w", encoding="utf-8") as f:
        json.dump(
            {
                "summary": summary,
                "classification_report": report_dict,
            },
            f,
            indent=2
        )

    # Also save a CSV-friendly report table
    # (classification_report dict -> DataFrame)
    df_rep = pd.DataFrame(report_dict).T
    df_rep.to_csv(os.path.join(cfg.OUT_DIR, "test_report_table.csv"), index=True)

    print("\nSaved test reports:")
    print(" -", os.path.join(cfg.OUT_DIR, "test_report.json"))
    print(" -", os.path.join(cfg.OUT_DIR, "test_report_table.csv"))

In [10]:
def main():
    cfg = CFG()
    ensure_dir(cfg.OUT_DIR)

    # quick directory print
    train_dir, val_dir, test_dir = resolve_split_paths(cfg)
    print("==== Paths ====")
    print("TRAIN:", train_dir, "->", os.listdir(train_dir))
    print("VALID:", val_dir,   "->", os.listdir(val_dir))
    print("TEST :", test_dir,  "->", os.listdir(test_dir))

    best = run_optuna_tuning(cfg)
    final_train_and_test(cfg, best)


if __name__ == "__main__":
    main()

==== Paths ====
TRAIN: /kaggle/input/datasets/carson650/wildfire/wildfire/train/images -> ['nofire', 'fire']
VALID: /kaggle/input/datasets/carson650/wildfire/wildfire/valid/images -> ['nofire', 'fire']
TEST : /kaggle/input/datasets/carson650/wildfire/wildfire/test -> ['nofire', 'fire']


[I 2026-03-12 09:51:47,466] A new study created in memory with name: flamevision_resnet50_ce


==== Dataset ====
BASE_DIR: /kaggle/input/datasets/carson650/wildfire/wildfire
train samples: 11198 | valid samples: 2000
classes: ['fire', 'nofire']
class_to_idx: {'fire': 0, 'nofire': 1}
class_weights (for CE): [0.965499997138977, 1.0369999408721924]
device: cuda

==== Optuna tuning start ====


  0%|          | 0/10 [00:00<?, ?it/s]

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth



  0%|          | 0.00/97.8M [00:00<?, ?B/s]
 12%|█▏        | 11.6M/97.8M [00:00<00:00, 121MB/s]
 29%|██▉       | 28.1M/97.8M [00:00<00:00, 151MB/s]
 48%|████▊     | 46.8M/97.8M [00:00<00:00, 171MB/s]
 65%|██████▍   | 63.1M/97.8M [00:00<00:00, 168MB/s]
100%|██████████| 97.8M/97.8M [00:00<00:00, 177MB/s]


[trial 000] epoch 01/10 loss=0.3518 val_acc=0.9470 val_recall(macro)=0.9371 val_f1(macro)=0.9427 lr=1.84e-04 time=76.1s
[trial 000] epoch 02/10 loss=0.1940 val_acc=0.9535 val_recall(macro)=0.9492 val_f1(macro)=0.9503 lr=1.84e-04 time=64.7s
[trial 000] epoch 03/10 loss=0.1597 val_acc=0.9545 val_recall(macro)=0.9476 val_f1(macro)=0.9511 lr=1.84e-04 time=64.5s
[trial 000] epoch 04/10 loss=0.0642 val_acc=0.9775 val_recall(macro)=0.9809 val_f1(macro)=0.9762 lr=1.84e-04 time=72.3s
[trial 000] epoch 05/10 loss=0.0313 val_acc=0.9925 val_recall(macro)=0.9905 val_f1(macro)=0.9920 lr=1.84e-04 time=72.6s
[trial 000] epoch 06/10 loss=0.0236 val_acc=0.9870 val_recall(macro)=0.9843 val_f1(macro)=0.9861 lr=1.84e-04 time=73.2s
[trial 000] epoch 07/10 loss=0.0197 val_acc=0.9870 val_recall(macro)=0.9867 val_f1(macro)=0.9861 lr=1.84e-04 time=72.8s
[trial 000] epoch 08/10 loss=0.0197 val_acc=0.9845 val_recall(macro)=0.9825 val_f1(macro)=0.9834 lr=1.84e-04 time=72.3s
[I 2026-03-12 10:01:17,436] Trial 0 fini